# Are we texting at work? (week 5, relationships)

Builds the Analysis 17 chart from `analysis-log.md`.

**Decisions, made with the student along the way (details in the log):**

- Proposition: on weekdays the group texts more during working hours (08:00–17:00) than on weekends at the same hours.
  Expected: the weekday share in 8–17 is about 20% higher (relative) than the weekend share. Falsified if it is not higher.
- Unit: one message. Media, deleted messages, locations and contact cards are **kept** (still activity on the phone).
- Weekend = Saturday, Sunday and public holidays (New Year, Easter Monday, King's Day, Ascension, Whit Monday, Christmas).
  Good Friday and Liberation Day stay weekdays.
- Main chart: pooled share of messages per hour, weekday vs weekend. Checks: average of daily shares, per weekday, during vs after the
  working-from-home period (`during` = up to 2022-09-17).

In [ ]:
import numpy as np
import pandas as pd

from wa_analyzer.data import load_own_chat

WORK_START, WORK_END = 8, 17  # hours -- 08:00 up to (not incl.) 17:00
RESTRICTIONS_END = pd.Timestamp("2022-09-17")  # analysis 1 table: restrictions fully ended
HOLIDAYS = pd.to_datetime(
    [
        # New Year's Day
        "2021-01-01", "2022-01-01", "2023-01-01", "2024-01-01", "2025-01-01", "2026-01-01",
        # Easter Monday
        "2021-04-05", "2022-04-18", "2023-04-10", "2024-04-01", "2025-04-21", "2026-04-06",
        # King's Day (moves to the 26th when the 27th is a Sunday)
        "2021-04-27", "2022-04-27", "2023-04-27", "2024-04-27", "2025-04-26", "2026-04-27",
        # Ascension Day
        "2021-05-13", "2022-05-26", "2023-05-18", "2024-05-09", "2025-05-29", "2026-05-14",
        # Whit Monday
        "2021-05-24", "2022-06-06", "2023-05-29", "2024-05-20", "2025-06-09", "2026-05-25",
        # Christmas
        "2020-12-25", "2020-12-26", "2021-12-25", "2021-12-26", "2022-12-25", "2022-12-26",
        "2023-12-25", "2023-12-26", "2024-12-25", "2024-12-26", "2025-12-25", "2025-12-26",
    ]
)

own = load_own_chat()
# same fix as 06-12: `timestamp` is stored as strings; the `+00:00` is only a label
own["timestamp"] = pd.to_datetime(own["timestamp"], utc=True).dt.tz_localize(None)
own = own.sort_values("timestamp", kind="stable").reset_index(drop=True)
own["day"] = own["timestamp"].dt.normalize()
own["hour"] = own["timestamp"].dt.hour
own["weekday_name"] = own["timestamp"].dt.day_name()
own["day_type"] = np.where(
    (own["timestamp"].dt.dayofweek >= 5) | own["day"].isin(HOLIDAYS), "weekend", "weekday"
)
own["period"] = np.where(own["timestamp"] < RESTRICTIONS_END, "during", "after")
own["work_hours"] = own["hour"].between(WORK_START, WORK_END - 1)
print(f"{len(own):,} messages, {own['author'].nunique()} authors, {own['day'].min():%Y-%m-%d} to {own['day'].max():%Y-%m-%d}")

## Stage 3 — what is there to compare?

Counts only, no shares yet: how many messages and active days each group has, and how many holidays moved to `weekend`.

In [ ]:
holidays_in_data = own.loc[own["day"].isin(HOLIDAYS), "day"].nunique()
weekday_holidays = own.loc[own["day"].isin(HOLIDAYS) & (own["timestamp"].dt.dayofweek < 5), "day"].nunique()
print(f"{holidays_in_data} holidays with messages, {weekday_holidays} of them on a Mon-Fri (moved to weekend)")

overview = own.groupby(["period", "day_type"]).agg(messages=("message", "size"), active_days=("day", "nunique"))
overview["messages_per_day"] = overview["messages"] / overview["active_days"]
overview.round(1)

In [ ]:
# per weekday: messages and active days (the per-weekday check needs enough of both)
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
own.groupby("weekday_name").agg(messages=("message", "size"), active_days=("day", "nunique")).reindex(order)

In [ ]:
# messages per hour, weekday vs weekend -- raw counts; the shape, not yet the claim
pd.crosstab(own["hour"], own["day_type"])

## Stage 4 — first draft of the main chart

Pooled share of messages per hour (all messages of that day type together), every hour 0–23 on the axis even at 0.
`active_days` above counts only days with at least one message, so it is not used here.

In [ ]:
import matplotlib.pyplot as plt

DAY_TYPE_COLORS = {"weekday": "#2a78d6", "weekend": "#eb6834"}


def hourly_shares(frame, group):
    counts = pd.crosstab(frame["hour"], frame[group]).reindex(range(24), fill_value=0)
    return counts / counts.sum() * 100


shares = hourly_shares(own, "day_type")
work_share = shares.loc[WORK_START : WORK_END - 1].sum()
print(f"share of messages in {WORK_START:02d}-{WORK_END:02d}h: weekday {work_share['weekday']:.1f}%, weekend {work_share['weekend']:.1f}%")
print(f"weekday / weekend = {work_share['weekday'] / work_share['weekend']:.2f} (expected about 1.20)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.axvspan(WORK_START - 0.5, WORK_END - 0.5, color="0.92", zorder=0)
ax.text((WORK_START + WORK_END - 1) / 2, ax.get_ylim()[1], "working hours 08-17", ha="center", va="top", color="0.35")
for day_type, color in DAY_TYPE_COLORS.items():
    ax.plot(shares.index, shares[day_type], color=color, linewidth=2, marker="o", markersize=4, label=day_type)
    ax.annotate(day_type, (23, shares.loc[23, day_type]), xytext=(6, 0), textcoords="offset points", va="center", color="0.2")
ax.set_xticks(range(24))
ax.set_xlim(-0.5, 24.5)
ax.set_ylim(bottom=0)
ax.set_xlabel("hour of day")
ax.set_ylabel("share of that day type's messages (%)")
ax.set_title("Draft: share of messages per hour, weekdays vs weekends (holidays = weekend), 2020-2026", loc="left")
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="0.9")
fig.tight_layout()
fig.savefig("work-hours-draft.png", dpi=150)

## During vs after working from home

Expectation recorded before this cell was run: the weekday share in 08–17 was higher **during** (up to 2022-09-17) than **after**.
Weekends stay in as the baseline within each period: if both day types shift the same way, it is not about work.

In [ ]:
periods = ["during", "after"]
period_shares = {period: hourly_shares(own[own["period"] == period], "day_type") for period in periods}
work_by_period = pd.DataFrame({period: shares.loc[WORK_START : WORK_END - 1].sum() for period, shares in period_shares.items()}).T
work_by_period["weekday / weekend"] = work_by_period["weekday"] / work_by_period["weekend"]
work_by_period.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
titles = {"during": "During working from home (to 2022-09-17)", "after": "After (from 2022-09-17)"}
for ax, period in zip(axes, periods):
    ax.axvspan(WORK_START - 0.5, WORK_END - 0.5, color="0.92", zorder=0)
    for day_type, color in DAY_TYPE_COLORS.items():
        ax.plot(range(24), period_shares[period][day_type], color=color, linewidth=2, marker="o", markersize=4, label=day_type)
    ax.set_title(titles[period], loc="left")
    ax.set_xticks(range(0, 24, 2))
    ax.set_xlabel("hour of day")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="0.9")
axes[0].set_ylabel("share of that day type's messages (%)")
axes[0].set_ylim(bottom=0)
axes[0].legend(frameon=False, loc="upper left")
fig.suptitle("Draft: share of messages per hour, working hours 08-17 shaded", x=0.01, ha="left")
fig.tight_layout()
fig.savefig("work-hours-period-draft.png", dpi=150)

## Stage 6 — checks (chosen in stage 2, before seeing any shares)

Rule, same as the stage 1 falsification: a check **changes the conclusion** only if the weekday share in 08–17 comes out
higher than the weekend share, with a 95% interval (resampling days) that stays above a ratio of 1.

| Check | What it guards against |
|---|---|
| Interval on the pooled ratio (resample days, 1,000×) | the main number being luck |
| Average of daily shares (each day weighs once) | a few busy days setting the pooled share |
| Per weekday, Mon–Fri vs weekend | one workday behaving differently, hidden in the weekday average |
| During vs after, on both bases | the working-from-home result resting on busy days |

In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)

per_day = own.groupby(["day", "day_type", "period", "weekday_name"]).agg(
    messages=("work_hours", "size"), in_work_hours=("work_hours", "sum")
).reset_index()
per_day["work_share"] = per_day["in_work_hours"] / per_day["messages"] * 100


def interval(days, statistic, draws=1000):
    """95% interval of statistic(days) from resampling days within each day type."""
    groups = [part for _, part in days.groupby("day_type")]
    values = [
        statistic(pd.concat([part.iloc[rng.integers(0, len(part), len(part))] for part in groups]))
        for _ in range(draws)
    ]
    return np.percentile(values, [2.5, 97.5])


def pooled_ratio(days):
    totals = days.groupby("day_type")[["in_work_hours", "messages"]].sum()
    share = totals["in_work_hours"] / totals["messages"]
    return share["weekday"] / share["weekend"]


def daily_ratio(days):
    share = days.groupby("day_type")["work_share"].mean()
    return share["weekday"] / share["weekend"]


def summary(days, label):
    pooled_lo, pooled_hi = interval(days, pooled_ratio)
    daily_lo, daily_hi = interval(days, daily_ratio)
    return {
        "version": label,
        "weekday_days": int((days["day_type"] == "weekday").sum()),
        "weekend_days": int((days["day_type"] == "weekend").sum()),
        "pooled_ratio": pooled_ratio(days),
        "pooled_95": f"{pooled_lo:.2f}-{pooled_hi:.2f}",
        "daily_ratio": daily_ratio(days),
        "daily_95": f"{daily_lo:.2f}-{daily_hi:.2f}",
    }


checks = [summary(per_day, "all days")]
checks += [summary(per_day[per_day["period"] == period], period) for period in ["during", "after"]]
pd.DataFrame(checks).round(2)

In [ ]:
# each workday against all weekend days (holidays stay weekend)
weekend_days = per_day[per_day["day_type"] == "weekend"]
by_weekday = [
    summary(pd.concat([per_day[(per_day["day_type"] == "weekday") & (per_day["weekday_name"] == name)], weekend_days]), name)
    for name in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
]
pd.DataFrame(by_weekday).round(2)

In [ ]:
# how many messages a day has matters for the daily average: a 1-message day is 0% or 100%
per_day.groupby("day_type")["messages"].describe(percentiles=[0.25, 0.5, 0.75]).round(1)

In [ ]:
# the second question directly: weekday share in 08-17, during minus after (percentage points)
weekdays = per_day[per_day["day_type"] == "weekday"]


def during_minus_after(days, basis):
    if basis == "pooled":
        totals = days.groupby("period")[["in_work_hours", "messages"]].sum()
        share = totals["in_work_hours"] / totals["messages"] * 100
    else:
        share = days.groupby("period")["work_share"].mean()
    return share["during"] - share["after"]


rows = []
for basis in ["pooled", "daily"]:
    parts = [part for _, part in weekdays.groupby("period")]
    draws = [
        during_minus_after(pd.concat([part.iloc[rng.integers(0, len(part), len(part))] for part in parts]), basis)
        for _ in range(1000)
    ]
    lo, hi = np.percentile(draws, [2.5, 97.5])
    rows.append({"basis": basis, "during_minus_after_pp": during_minus_after(weekdays, basis), "95": f"{lo:.1f} to {hi:.1f}"})
pd.DataFrame(rows).round(1)

## Final chart

Story A (student): the 08–17 share is the same on weekdays and weekends. B (earlier start, later evenings, weekend nights)
stays visible as context with light highlights only -- it was not tested. During/after goes in the footnote.

In [ ]:
TITLE = "Office hours don't change the time we text"  # story A, student's wording
HIGHLIGHTS = [  # (first hour, last hour, label) -- context (story B), not tested
    (0, 2, "weekend\nnights"),
    (6, 7, "weekdays\nstart earlier"),
    (21, 23, "weekday evenings\nrun later"),
]
FOOTNOTE = (
    f"Data: all {len(own):,} messages in the friends' group chat, 2020-08-12 to 2026-09-08, 9 people; "
    "media and deleted messages count as messages. Weekend = Saturday, Sunday and public holidays.\n"
    f"Share of messages in 08-17: weekdays {work_share['weekday']:.0f}%, weekends {work_share['weekend']:.0f}% "
    "(plausible range for weekdays 44-55%, resampling days). Same result per weekday and with every day weighted once.\n"
    "During working from home (to 2022-09-17) and after: weekday share 48% vs 49%, no clear change."
)

fig, ax = plt.subplots(figsize=(11, 6))
ax.axvspan(WORK_START - 0.5, WORK_END - 0.5, color="0.93", zorder=0)
for first, last, label in HIGHLIGHTS:
    ax.axvspan(first - 0.5, last + 0.5, color="#fbe3d6" if first == 0 else "#dbe9fa", alpha=0.6, zorder=0)
for day_type, color in DAY_TYPE_COLORS.items():
    ax.plot(shares.index, shares[day_type], color=color, linewidth=2, marker="o", markersize=4, label=day_type)
    ax.annotate(day_type, (23, shares.loc[23, day_type]), xytext=(6, 0), textcoords="offset points", va="center", color="0.2")
top = shares.to_numpy().max() * 1.18
ax.set_ylim(0, top)
ax.text((WORK_START + WORK_END - 1) / 2, top * 0.97, "working hours 08-17\nweekdays 49%, weekends 51% of messages", ha="center", va="top", color="0.25")
for first, last, label in HIGHLIGHTS:
    ax.text((first + last) / 2, top * 0.97, label, ha="center", va="top", color="0.45", fontsize=8)
ax.set_xticks(range(24))
ax.set_xlim(-0.5, 24.8)
ax.set_xlabel("hour of day")
ax.set_ylabel("share of that day type's messages (%)")
ax.set_title(TITLE, loc="left", fontsize=13)
ax.legend(frameon=False, loc="center left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="0.9")
fig.text(0.01, 0.01, FOOTNOTE, fontsize=7.5, color="0.35", va="bottom")
fig.tight_layout(rect=(0, 0.1, 1, 1))
fig.savefig("work-hours-final.png", dpi=150)